# Minicurso — Estrutura eletrônica, reatividade e descritores quânticos em sistemas biológicos

## PRIMoRDiA 1.50 + OOCCuPY + pDynamo3 + MOPAC

Este notebook foi organizado para o **Google Colab** e acompanha o material em PDF do minicurso. Ele tem dois usos:

1. **durante a aula:** instalar/testar o ambiente e executar os cálculos e análises do PRIMoRDiA;
2. **depois da aula:** reproduzir, com OOCCuPY/pDynamo3 e MOPAC, as etapas de preparação que geraram os dados das Atividades A e B.

### Organização didática

- **Parte 0 — Clones dos repositórios**: somente download do código e dos dados.
- **Parte 1 — Instalação dos programas**: dependências do sistema, R, pDynamo3, OOCCuPY, MOPAC, PRIMoRDiA e `py3Dmol`.
- **Parte 2 — Diagnóstico do ambiente**: cada programa é chamado/importado antes das atividades.
- **Parte 3 — OOCCuPY/pDynamo3 (reprodução opcional)**: preparação completa das Atividades A e B. **Não é necessário executar esta seção na sequência principal da aula.**
- **Parte 4 — Atividades do curso**: comandos planejados para execução em sala.
- **Parte 5 — Visualização 3D no Colab**: inspeção de PDBs e coloração pelo **B-factor**, inclusive quando ele for usado para armazenar valores de descritores.

> **Importante:** o Colab é efêmero. Se a sessão for reiniciada, arquivos em `/content` podem ser perdidos. Os dados originais continuam disponíveis no GitHub.


## 0. Clones dos repositórios

Nesta seção **não instalamos nada**. Apenas clonamos os repositórios necessários em caminhos previsíveis. A árvore atual do repositório do minicurso usa diretamente `Atividade_A/`, `Atividade_B/` e `Atividade_C/` na raiz.


In [ ]:
from pathlib import Path

CONTENT = Path("/content")
SRC = CONTENT / "software_src"
COURSE = CONTENT / "minicurso_xii_emmsb_primordia"

PDYNAMO_SRC = SRC / "pDynamo3"
OOCCUPY_SRC = SRC / "OOCCuPY"
PRIMORDIA_SRC = SRC / "PRIMoRDiA_1.50v"
MOPAC_SRC = SRC / "MOPAC"

SRC.mkdir(parents=True, exist_ok=True)

print("COURSE        =", COURSE)
print("PDYNAMO_SRC   =", PDYNAMO_SRC)
print("OOCCUPY_SRC   =", OOCCUPY_SRC)
print("PRIMORDIA_SRC =", PRIMORDIA_SRC)
print("MOPAC_SRC     =", MOPAC_SRC)

### 0.1 Repositório do minicurso e dados

In [ ]:
%%bash
set -e
rm -rf /content/minicurso_xii_emmsb_primordia
git clone --depth 1 https://github.com/bardenChem/minicurso_xii_emmsb_primordia.git /content/minicurso_xii_emmsb_primordia

### 0.2 Repositório do PRIMoRDiA 1.50

In [ ]:
%%bash
set -e
rm -rf /content/software_src/PRIMoRDiA_1.50v
git clone --depth 1 https://github.com/bardenChem/PRIMoRDiA_1.50v.git /content/software_src/PRIMoRDiA_1.50v

### 0.3 Repositório do OOCCuPY

In [ ]:
%%bash
set -e
rm -rf /content/software_src/OOCCuPY
git clone --depth 1 https://github.com/bardenChem/OOCCuPY.git /content/software_src/OOCCuPY

### 0.4 Repositório oficial do pDynamo3

In [ ]:
%%bash
set -e
rm -rf /content/software_src/pDynamo3
git clone --depth 1 https://github.com/pdynamo/pDynamo3.git /content/software_src/pDynamo3

### 0.5 Repositório oficial do MOPAC

In [ ]:
%%bash
set -e
rm -rf /content/software_src/MOPAC
git clone --depth 1 https://github.com/openmopac/MOPAC.git /content/software_src/MOPAC

### 0.6 Conferência da árvore do minicurso

In [ ]:
from pathlib import Path

for activity in ["Atividade_A", "Atividade_B", "Atividade_C"]:
    p = COURSE / activity
    print(f"\n[{activity}]  existe={p.is_dir()}  caminho={p}")
    if p.is_dir():
        for f in sorted(p.iterdir())[:30]:
            print("  ", f.name)

assert (COURSE / "Atividade_A").is_dir()
assert (COURSE / "Atividade_B").is_dir()
assert (COURSE / "Atividade_C").is_dir()
print("\nÁrvore principal do curso: OK")

# 1. Instalação dos programas e bibliotecas

Agora instalamos as dependências. Esta seção está separada dos clones para facilitar diagnóstico: se um build falhar, os repositórios continuam disponíveis e não precisam ser baixados novamente.


### 1.1 Dependências do sistema, compiladores, R e PyMOL

In [ ]:
%%bash
set -e
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y \
  build-essential cmake git gcc g++ gfortran make \
  python3-dev cython3 python3-yaml python3-numpy \
  libeigen3-dev libboost-iostreams-dev zlib1g-dev \
  libblas-dev liblapack-dev \
  r-base r-base-dev \
  pymol

### 1.2 Bibliotecas Python usadas no notebook

In [ ]:
%pip install -q py3Dmol pandas matplotlib

### 1.3 Pacotes de R

Os scripts do PRIMoRDiA e as análises do minicurso usam pacotes gráficos/estatísticos. Instalamos por `apt` os pacotes disponíveis na distribuição e usamos CRAN para completar o ambiente. O script de densidade conjunta da trajetória da RTA usa especificamente `ggplot2` e `MASS`.


In [ ]:
%%bash
set -e
DEBIAN_FRONTEND=noninteractive apt-get install -y \
  r-cran-ggplot2 r-cran-ggpubr r-cran-pheatmap \
  r-cran-factominer r-cran-factoextra r-cran-corrplot r-cran-caret \
  r-cran-mass

Rscript -e 'pkgs <- c("ggplot2","ggpubr","pheatmap","FactoMineR","factoextra","corrplot","caret","MASS","KRLS"); \
inst <- rownames(installed.packages()); miss <- setdiff(pkgs, inst); \
if(length(miss)>0) install.packages(miss, repos="https://cloud.r-project.org"); \
miss2 <- pkgs[!vapply(pkgs, requireNamespace, logical(1), quietly=TRUE)]; \
if(length(miss2)>0) stop(paste("Pacotes ausentes:", paste(miss2, collapse=", "))); \
cat("Pacotes R OK:\n", paste(pkgs, collapse=", "), "\n")' 

### 1.4 Instalação/compilação do pDynamo3

O pDynamo3 possui módulos em C/Cython. O procedimento oficial compila o código a partir do diretório `installation`.


In [ ]:
%%bash
set -e
cd /content/software_src/pDynamo3/installation
python3 Install.py -f

### 1.5 Configurar o ambiente do pDynamo3 dentro do kernel do Colab

In [ ]:
import os, sys
from pathlib import Path

PDYNAMO_HOME = str(PDYNAMO_SRC)
PDYNAMO_SCRATCH = "/content/pdynamo_scratch"
Path(PDYNAMO_SCRATCH).mkdir(parents=True, exist_ok=True)

os.environ["PDYNAMO3_HOME"] = PDYNAMO_HOME
os.environ["PDYNAMO3_SCRATCH"] = PDYNAMO_SCRATCH
os.environ["PDYNAMO3_PARAMETERS"] = f"{PDYNAMO_HOME}/parameters"
os.environ["PDYNAMO3_PYTHONCOMMAND"] = "python3"
os.environ["PDYNAMO3_STYLE"] = f"{PDYNAMO_HOME}/parameters/ccsStyleSheets/defaultStyle.css"

if PDYNAMO_HOME not in sys.path:
    sys.path.insert(0, PDYNAMO_HOME)

print("Variáveis do pDynamo3 configuradas para esta sessão.")

### 1.6 Instalar OOCCuPY a partir do clone

In [ ]:
%pip install -q -e /content/software_src/OOCCuPY

### 1.7 Compilar e instalar MOPAC

Usamos o repositório oficial moderno do MOPAC. A instalação fica em `/content/mopac`, separada do código-fonte.


In [ ]:
%%bash
set -e
rm -rf /content/software_src/MOPAC/build /content/mopac
cmake -S /content/software_src/MOPAC \
      -B /content/software_src/MOPAC/build \
      -DCMAKE_BUILD_TYPE=Release \
      -DCMAKE_INSTALL_PREFIX=/content/mopac
cmake --build /content/software_src/MOPAC/build --parallel 2
cmake --install /content/software_src/MOPAC/build

In [ ]:
import os
os.environ["PATH"] = "/content/mopac/bin:" + os.environ.get("PATH", "")
print("PATH atualizado com /content/mopac/bin")

### 1.8 Compilar o PRIMoRDiA 1.50

In [ ]:
%%bash
set -e
rm -rf /content/software_src/PRIMoRDiA_1.50v/build
cmake -S /content/software_src/PRIMoRDiA_1.50v \
      -B /content/software_src/PRIMoRDiA_1.50v/build \
      -DCMAKE_BUILD_TYPE=Release
cmake --build /content/software_src/PRIMoRDiA_1.50v/build --parallel 2

In [ ]:
from pathlib import Path
import os

candidates = [
    PRIMORDIA_SRC / "PRIMoRDiA_1.50v",
    PRIMORDIA_SRC / "build" / "PRIMoRDiA_1.50v",
]
PRIMORDIA = next((p for p in candidates if p.exists()), None)
if PRIMORDIA is None:
    raise FileNotFoundError("Executável PRIMoRDiA_1.50v não encontrado após o build.")
PRIMORDIA = str(PRIMORDIA)
print("PRIMORDIA =", PRIMORDIA)

# 2. Testes e diagnóstico dos programas

Faça estes testes **antes** de iniciar a aula prática. Eles foram separados para que um problema seja identificado imediatamente.


### 2.1 pDynamo3 — importação mínima

In [ ]:
from pCore import logFile
from pMolecule import System

print("pDynamo3 importado com sucesso")
print("pCore.logFile:", type(logFile).__name__)
print("pMolecule.System:", System)

### 2.2 OOCCuPY — help e configuração

In [ ]:
import subprocess, shutil

ooccupy_bin = shutil.which("ooccupy")
print("ooccupy:", ooccupy_bin)
if not ooccupy_bin:
    raise RuntimeError("OOCCuPY não foi encontrado no PATH")

p = subprocess.run([ooccupy_bin, "--help"], capture_output=True, text=True)
print((p.stdout or p.stderr)[:5000])

In [ ]:
p = subprocess.run([ooccupy_bin, "config", "--show"], capture_output=True, text=True)
print((p.stdout or p.stderr)[:5000])

#### Help específico do wrapper pDynamo

Este comando é útil para confirmar as opções aceitas pela versão instalada antes de executar as etapas opcionais A/B.

In [ ]:
p = subprocess.run([ooccupy_bin, "pdynamo", "--help"], capture_output=True, text=True)
print((p.stdout or p.stderr)[:5000])

### 2.3 MOPAC — localização e chamada sem arquivo de entrada

In [ ]:
import subprocess, shutil

MOPAC_BIN = shutil.which("mopac")
print("MOPAC_BIN =", MOPAC_BIN)
if not MOPAC_BIN:
    raise RuntimeError("MOPAC não foi encontrado no PATH")

# MOPAC tradicionalmente mostra mensagem de uso/entrada quando chamado sem arquivo.
p = subprocess.run([MOPAC_BIN], capture_output=True, text=True, timeout=20)
msg = (p.stdout or "") + "\n" + (p.stderr or "")
print(msg[:5000])
print("return code:", p.returncode)

### 2.4 PRIMoRDiA — help

In [ ]:
p = subprocess.run([PRIMORDIA, "--help"], capture_output=True, text=True)
print((p.stdout or p.stderr)[:7000])
print("return code:", p.returncode)

### 2.5 Diagnóstico final do ambiente

In [ ]:
import os, shutil

checks = {
    "Dados do curso": COURSE.is_dir(),
    "Atividade_A": (COURSE / "Atividade_A").is_dir(),
    "Atividade_B": (COURSE / "Atividade_B").is_dir(),
    "Atividade_C": (COURSE / "Atividade_C").is_dir(),
    "pDynamo3": "pCore" in sys.modules,
    "OOCCuPY": shutil.which("ooccupy") is not None,
    "MOPAC": shutil.which("mopac") is not None,
    "PRIMoRDiA": Path(PRIMORDIA).exists(),
    "Rscript": shutil.which("Rscript") is not None,
    "PyMOL": shutil.which("pymol") is not None,
}

for name, ok in checks.items():
    print(f"{name:18s}: {'OK' if ok else 'FALHOU'}")

if not all(checks.values()):
    raise RuntimeError("Há componentes ausentes. Corrija o ambiente antes de seguir.")

# 3. OOCCuPY/pDynamo3 — reprodução opcional das preparações

**Esta seção é material para executar depois da aula ou em um ensaio controlado.** Ela documenta como os dados das Atividades A e B são produzidos. Os comandos estão prontos, mas ficam protegidos por flags para evitar iniciar acidentalmente cálculos mais longos.

O OOCCuPY usa a forma:

```bash
ooccupy pdynamo --input arquivo.inp --proj-folder pasta_de_saida
```

Cada etapa usa arquivos gerados pela anterior.


### 3.1 Utilitário: ajustar automaticamente o caminho do MOPAC nos inputs

In [ ]:
from pathlib import Path
import re, shutil

def patch_mopac_path(input_file):
    input_file = Path(input_file)
    mopac_bin = shutil.which("mopac")
    if not mopac_bin:
        raise RuntimeError("MOPAC não encontrado no PATH")
    text = input_file.read_text()
    if "#MOPAC_PATH" in text:
        text = re.sub(r"(?m)^#MOPAC_PATH\s+.*$", f"#MOPAC_PATH {mopac_bin}", text)
        input_file.write_text(text)
        print("MOPAC_PATH atualizado em", input_file)
        print(" ->", mopac_bin)
    else:
        print("O input não contém #MOPAC_PATH:", input_file)


## 3A. Atividade A — RTA: preparação, QM/MM e refinamento

A árvore atual contém os seis inputs diretamente em `Atividade_A/`. O `step_06_md_refinement.inp` atual usa a trajetória `MD_sampled.ptGeo`, uma região centrada na água selecionada e `XNBINS 51` para o refinamento eletrônico.

**A dinâmica QM/MM e o refinamento MOPAC podem ser demorados.** Por isso, esta seção não faz parte da sequência obrigatória do curso.


In [ ]:
A_DIR = COURSE / "Atividade_A"
for f in sorted(A_DIR.glob("step_*.inp")):
    print(f.name)


In [ ]:
RUN_OPTIONAL_A = False  # mude para True apenas quando quiser reproduzir a preparação

if RUN_OPTIONAL_A:
    import subprocess
    patch_mopac_path(A_DIR / "step_06_md_refinement.inp")
    commands = [
        ["ooccupy","pdynamo","--input","step_01_load_amber_info.inp","--proj-folder","step01"],
        ["ooccupy","pdynamo","--input","step_02_geo_opt_mm_all.inp","--proj-folder","step02"],
        ["ooccupy","pdynamo","--input","step_03_pruned_fix_opt.inp","--proj-folder","step03"],
        ["ooccupy","pdynamo","--input","step_04_qmmm_set_opt.inp","--proj-folder","step04"],
        ["ooccupy","pdynamo","--input","step_05_md_qmmm.inp","--proj-folder","step05"],
        ["ooccupy","pdynamo","--input","step_06_md_refinement.inp","--proj-folder","step06"],
    ]
    for cmd in commands:
        print("\n$", " ".join(cmd))
        subprocess.run(cmd, cwd=A_DIR, check=True)
else:
    print("Etapas OOCCuPY da Atividade A NÃO executadas (RUN_OPTIONAL_A=False).")

## 3B. Atividade B — TIM: otimização QM/MM, scan 1D e refinamento

A Atividade B usa a triose-fosfato isomerase. A sequência atual prepara o sistema, recorta/fixa a região externa, otimiza o QM/MM, executa um `Relaxed_Surface_Scan` 1D e depois prepara o refinamento MOPAC.


In [ ]:
B_DIR = COURSE / "Atividade_B"
for f in sorted(B_DIR.glob("step_*.inp")):
    print(f.name)


In [ ]:
RUN_OPTIONAL_B = False  # mude para True se quiser reproduzir a preparação

if RUN_OPTIONAL_B:
    import subprocess
    patch_mopac_path(B_DIR / "step_05_mopac_refinement.inp")
    commands = [
        ["ooccupy","pdynamo","--input","step_01_load_data.inp","--proj-folder","step01"],
        ["ooccupy","pdynamo","--input","step_02_mm_opt.inp","--proj-folder","step02"],
        ["ooccupy","pdynamo","--input","step_03_qmmm_opt.inp","--proj-folder","step03"],
        ["ooccupy","pdynamo","--input","step_04_qmmm_scan.inp","--proj-folder","step04"],
        ["ooccupy","pdynamo","--input","step_05_mopac_refinement.inp","--proj-folder","step05"],
    ]
    for cmd in commands:
        print("\n$", " ".join(cmd))
        subprocess.run(cmd, cwd=B_DIR, check=True)
else:
    print("Etapas OOCCuPY da Atividade B NÃO executadas (RUN_OPTIONAL_B=False).")

# 4. Parte do curso — comandos executados em aula

A partir daqui está a sequência principal. Os cálculos mais demorados de QM/MM ficam fora do caminho crítico da aula. Em A e B, os inputs do PRIMoRDiA são montados a partir dos arquivos realmente disponíveis, para reforçar a leitura da estrutura de nomes e evitar caminhos antigos.


## 4.0 Atividade Zero — checkpoint do ambiente

In [ ]:
print("PRIMoRDiA:", PRIMORDIA)
print("OOCCuPY   :", shutil.which("ooccupy"))
print("MOPAC     :", shutil.which("mopac"))
print("Rscript   :", shutil.which("Rscript"))
print("Curso     :", COURSE)

for activity in ["Atividade_A", "Atividade_B", "Atividade_C"]:
    print(activity, "->", (COURSE/activity).is_dir())

## 4A. Atividade A — descritores ao longo de uma trajetória QM/MM da RTA

### Objetivo

Usar uma sequência de estruturas eletrônicas refinadas para calcular descritores condensados por resíduo, acompanhar sua variação ao longo da trajetória e comparar uma seleção baseada em **reatividade** com uma seleção baseada apenas em **geometria/estrutura**.

O input `trajectory` será construído em aula. O PRIMoRDiA deverá receber pares consistentes de arquivo eletrônico + PDB de cada frame.


### 4A.1 Inspecionar o material disponível

In [ ]:
A_DIR = COURSE / "Atividade_A"

qm_files = sorted(list(A_DIR.rglob("*.aux")) + list(A_DIR.rglob("*.aux.gz")))
pdb_files = sorted(A_DIR.rglob("*.pdb"))

print("Arquivos eletrônicos (.aux/.aux.gz):", len(qm_files))
print("PDBs encontrados:", len(pdb_files))
print("\nPrimeiros arquivos eletrônicos:")
for f in qm_files[:20]: print(" ", f.relative_to(A_DIR))
print("\nPrimeiros PDBs:")
for f in pdb_files[:20]: print(" ", f.relative_to(A_DIR))

if not qm_files:
    print("\nATENÇÃO: nenhum AUX refinado está atualmente dentro de Atividade_A.")
    print("Se os checkpoints forem adicionados ao repositório, basta rodar novamente esta célula.")
    print("Alternativamente, execute a seção opcional 3A para produzi-los.")

### 4A.2 Identificar o padrão de nomes

Confirme em aula:

- número total de frames;
- índice inicial (`0` ou `1`);
- prefixo comum dos arquivos;
- extensão (`.aux`, `.aux.gz`, etc.);
- PDB correspondente a cada estrutura eletrônica.

O refinamento atual da Atividade A está configurado para **51 bins**, portanto 51 é a expectativa natural se todos os pontos forem produzidos.


In [ ]:
import re
from collections import defaultdict
from pathlib import Path

def base_without_qm_extension(p: Path):
    name = p.name
    for ext in [".aux.gz", ".aux", ".mgf", ".out"]:
        if name.endswith(ext):
            return name[:-len(ext)], ext
    return p.stem, p.suffix

def summarize_numbered_series(root):
    root = Path(root)
    groups = defaultdict(list)
    qms = list(root.rglob("*.aux")) + list(root.rglob("*.aux.gz"))
    for f in qms:
        base, ext = base_without_qm_extension(f)
        m = re.match(r"^(.*?)(\d+)$", base)
        if not m:
            continue
        prefix, idx = m.group(1), int(m.group(2))
        groups[(f.parent, prefix, ext)].append(idx)
    rows = []
    for (parent,prefix,ext), idxs in groups.items():
        idxs = sorted(set(idxs))
        rows.append({
            "directory": parent,
            "prefix": prefix,
            "extension": ext,
            "count": len(idxs),
            "start": min(idxs),
            "end": max(idxs),
            "indices": idxs,
        })
    return sorted(rows, key=lambda x: x["count"], reverse=True)

A_series = summarize_numbered_series(A_DIR)
for s in A_series[:10]:
    print({k:(str(v.relative_to(A_DIR)) if k=="directory" else v) for k,v in s.items() if k != "indices"})

### 4A.3 Montar o input `trajectory`

In [ ]:
# Preenchimento preferencial: usar a maior série detectada automaticamente.
# Se necessário, altere manualmente as variáveis abaixo após a inspeção.

if A_series:
    s = A_series[0]
    A_QM_DIR = s["directory"]
    A_PREFIX = s["prefix"]
    A_QM_EXT = s["extension"]
    A_NFRAMES = s["count"]
    A_START = s["start"]
else:
    A_QM_DIR = A_DIR
    A_PREFIX = "frame"       # EDITAR quando os arquivos refinados estiverem disponíveis
    A_QM_EXT = ".aux"
    A_NFRAMES = 51
    A_START = 0

print("A_QM_DIR =", A_QM_DIR)
print("A_PREFIX =", A_PREFIX)
print("A_QM_EXT =", A_QM_EXT)
print("A_NFRAMES=", A_NFRAMES)
print("A_START  =", A_START)

# O PDB deve seguir o mesmo prefixo numerado. O input é salvo na mesma pasta da série.
A_INPUT = Path(A_QM_DIR) / "primordia_rta_trajectory.input"
text = f"""#RT trajectory
#PR gridsize 0 threads 4 band_method BD bandgap 5 eband 1 Rscript
#TRJ frames {A_NFRAMES} start {A_START}
#TRJ residues 173 176 260 327
3 {A_PREFIX}{A_QM_EXT} false {A_PREFIX}.pdb mopac 0 0 0 0
"""
A_INPUT.write_text(text)
print(text)
print("Input salvo em:", A_INPUT)

> **Checkpoint didático:** antes de rodar, verifique se o arquivo `prefixo0.pdb`/`prefixo1.pdb` realmente existe no mesmo diretório. O PRIMoRDiA precisa da mesma geometria e ordem de átomos no arquivo eletrônico e no PDB.

In [ ]:
# Verificação simples do primeiro par esperado
first_qm = Path(A_QM_DIR) / f"{A_PREFIX}{A_START}{A_QM_EXT}"
first_pdb = Path(A_QM_DIR) / f"{A_PREFIX}{A_START}.pdb"
print("QM :", first_qm, "exists=", first_qm.exists())
print("PDB:", first_pdb, "exists=", first_pdb.exists())

### 4A.4 Executar o PRIMoRDiA

In [ ]:
RUN_A_PRIMORDIA = first_qm.exists() and first_pdb.exists()

if RUN_A_PRIMORDIA:
    cmd = [PRIMORDIA, "-f", A_INPUT.name, "-np", "4", "-verbose"]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=A_QM_DIR, check=True)
    log = Path(A_QM_DIR) / "primordia.log"
    if log.exists():
        print("\n--- final do primordia.log ---")
        print("\n".join(log.read_text(errors="ignore").splitlines()[-50:]))
else:
    print("Execução pulada: o primeiro par AUX/PDB não está disponível.")

### 4A.5 Conferir outputs e scripts R

In [ ]:
A_outputs = Path(A_QM_DIR)
print(".rslrd:", len(list(A_outputs.glob("*.rslrd"))))
print("Scripts R gerados:")
for f in sorted(A_outputs.glob("*.R")):
    print(" ", f.name)

for name in ["residues_data_frames", "residues_data_stat", "protein_data_stat"]:
    p = A_outputs / name
    print(f"{name:24s} exists={p.exists()}")
    if p.exists():
        print("  ", "\n   ".join(p.read_text(errors="ignore").splitlines()[:4]))

### 4A.6 Script de densidade conjunta para seleção de frames

A versão atual do repositório do PRIMoRDiA contém:

`PRIMoRDiA_1.50v/scripts/primordia_joint_density_interactive.R`

Esse script lê `residues_data_frames`, permite escolher dois pares **resíduo + descritor**, estima a densidade de probabilidade 2D (KDE) e retorna os frames reais mais próximos do máximo da distribuição.

No terminal ele pode ser usado de forma interativa. No Colab, a entrada interativa de um `Rscript` pode ser menos confortável; por isso abaixo criamos também uma versão **Colab-friendly**, em que as escolhas são feitas por variáveis.


In [ ]:
R_TEMPLATE = PRIMORDIA_SRC / "scripts" / "primordia_joint_density_interactive.R"
print("Template R:", R_TEMPLATE)
print("exists:", R_TEMPLATE.exists())

if R_TEMPLATE.exists():
    print("\n".join(R_TEMPLATE.read_text(errors="ignore").splitlines()[:25]))

In [ ]:
# Criar uma cópia contendo apenas as funções, sem o bloco Main interativo.
# Isso permite chamá-las diretamente em um script R não interativo no Colab.

R_FUNCTIONS = A_outputs / "primordia_joint_density_functions.R"
if R_TEMPLATE.exists():
    src = R_TEMPLATE.read_text()
    marker = "#=======================================================================\n# Main\n#======================================================================="
    if marker in src:
        R_FUNCTIONS.write_text(src.split(marker)[0])
    else:
        raise RuntimeError("Marcador '# Main' não encontrado no template R")
    print("Funções copiadas para:", R_FUNCTIONS)
else:
    print("Template R não encontrado.")

#### Listar resíduos e descritores realmente presentes no arquivo

In [ ]:
R_DATA = A_outputs / "residues_data_frames"
R_LIST_SCRIPT = A_outputs / "list_rta_residues_descriptors.R"

if R_FUNCTIONS.exists() and R_DATA.exists():
    R_LIST_SCRIPT.write_text(f'''source("{R_FUNCTIONS.name}")
d <- read_primordia_residue_data("{R_DATA.name}")
cat("RESIDUES\\n")
print(sort(unique(d$residue)))
cat("\\nDESCRIPTORS\\n")
print(get_descriptor_names(d))
''')
    subprocess.run(["Rscript", R_LIST_SCRIPT.name], cwd=A_outputs, check=True)
else:
    print("Aguardando residues_data_frames para listar as opções.")

#### Executar uma análise de densidade conjunta no Colab

Depois da célula anterior, substitua os quatro valores abaixo **pelos rótulos exatamente mostrados**. Esta célula corresponde à lógica do script interativo, mas sem menus.


In [ ]:
# EDITE APÓS VER A LISTA PRODUZIDA ACIMA
R_RESIDUE_X = "EDITAR_RESIDUO_A"
R_DESCRIPTOR_X = "EDITAR_DESCRITOR_A"
R_RESIDUE_Y = "EDITAR_RESIDUO_B"
R_DESCRIPTOR_Y = "EDITAR_DESCRITOR_B"
R_NREP = 10

RUN_RTA_KDE = not any(x.startswith("EDITAR_") for x in [R_RESIDUE_X,R_DESCRIPTOR_X,R_RESIDUE_Y,R_DESCRIPTOR_Y])
print("RUN_RTA_KDE =", RUN_RTA_KDE)

In [ ]:
if RUN_RTA_KDE and R_FUNCTIONS.exists() and R_DATA.exists():
    runner = A_outputs / "run_joint_density_colab.R"
    runner.write_text(f'''source("{R_FUNCTIONS.name}")
d <- read_primordia_residue_data("{R_DATA.name}")
res <- analyze_joint_residue_density(
  residue_data=d,
  residue_x="{R_RESIDUE_X}", descriptor_x="{R_DESCRIPTOR_X}",
  residue_y="{R_RESIDUE_Y}", descriptor_y="{R_DESCRIPTOR_Y}",
  output_prefix="rta_joint_selected",
  number_representative_frames={R_NREP}
)
print(res$mode)
print(res$representative_frames)
''')
    subprocess.run(["Rscript", runner.name], cwd=A_outputs, check=True)
else:
    print("Defina os resíduos/descritores antes de executar esta análise.")

### 4A.7 Comparar com a amostragem estrutural

In [ ]:
structural_dir = A_DIR / "MD_sampled.ptGeo"
for f in sorted(structural_dir.glob("*.png")):
    print(f.name)

**Discussão:** um frame frequente em RMSD/raio de giração não precisa coincidir com o frame mais representativo de uma distribuição eletrônica. Registre separadamente o frame estrutural e o frame selecionado pela KDE dos descritores.


## 4B. Atividade B — caminho de reação da triose-fosfato isomerase

### Objetivo

Analisar a primeira etapa da reação da TIM a partir de um scan relaxado 1D. O modo `reaction` do PRIMoRDiA associa cada ponto do caminho à coordenada de reação e aos descritores dos resíduos monitorados.

A preparação OOCCuPY está documentada na Parte 3B. Se ela já tiver sido executada, os arquivos refinados estarão no diretório produzido pelo `step_05_mopac_refinement.inp`. Se forem fornecidos como checkpoint, a análise pode começar diretamente deles.


### 4B.1 Localizar arquivos refinados e o PDB da estrutura QM/MM

In [ ]:
B_DIR = COURSE / "Atividade_B"
B_qm = sorted(list(B_DIR.rglob("*.aux")) + list(B_DIR.rglob("*.aux.gz")))
B_pdb = sorted(B_DIR.rglob("*.pdb"))
print("AUX/AUX.GZ:", len(B_qm))
for f in B_qm[:30]: print(" ", f.relative_to(B_DIR))
print("\nPDBs:", len(B_pdb))
for f in B_pdb[:30]: print(" ", f.relative_to(B_DIR))

if not B_qm:
    print("\nNenhum refinamento eletrônico está atualmente dentro de Atividade_B.")
    print("Execute a Parte 3B ou use os checkpoints fornecidos para a aula.")

### 4B.2 Encontrar os átomos C02 e H02 no PDB

In [ ]:
def atom_serials_from_pdb(pdb_path, atom_names):
    found = []
    with open(pdb_path, errors="ignore") as fh:
        for line in fh:
            if line.startswith(("ATOM  ", "HETATM")):
                atom_name = line[12:16].strip()
                if atom_name in atom_names:
                    found.append({
                        "serial": int(line[6:11]),
                        "atom": atom_name,
                        "resname": line[17:20].strip(),
                        "chain": line[21:22].strip(),
                        "resid": line[22:26].strip(),
                    })
    return found

# Preferir o PDB gerado na otimização QM/MM, se existir.
candidate_pdbs = list(B_DIR.rglob("7tim_qcmm_opt.pdb")) + B_pdb
B_REFERENCE_PDB = candidate_pdbs[0] if candidate_pdbs else None
print("PDB de referência:", B_REFERENCE_PDB)
if B_REFERENCE_PDB:
    print(atom_serials_from_pdb(B_REFERENCE_PDB, {"C02","H02"}))

### 4B.3 Identificar a série numerada e montar o input `reaction`

In [ ]:
B_series = summarize_numbered_series(B_DIR)
for s in B_series[:10]:
    print({k:(str(v.relative_to(B_DIR)) if k=="directory" else v) for k,v in s.items() if k != "indices"})

In [ ]:
# Ajuste somente se a detecção automática não encontrar a série correta.
if B_series:
    s = B_series[0]
    B_QM_DIR = s["directory"]
    B_PREFIX = s["prefix"]
    B_QM_EXT = s["extension"]
    B_NPOINTS = s["count"]
    B_START = s["start"]
else:
    B_QM_DIR = B_DIR
    B_PREFIX = "frame"     # EDITAR conforme o checkpoint/refinamento
    B_QM_EXT = ".aux"
    B_NPOINTS = 12
    B_START = 0

# Preencha a partir da célula que listou C02/H02.
ATOM_C02 = None
ATOM_H02 = None
if B_REFERENCE_PDB:
    hits = atom_serials_from_pdb(B_REFERENCE_PDB, {"C02","H02"})
    c02 = [x["serial"] for x in hits if x["atom"] == "C02"]
    h02 = [x["serial"] for x in hits if x["atom"] == "H02"]
    if len(c02) == 1 and len(h02) == 1:
        ATOM_C02, ATOM_H02 = c02[0], h02[0]

print("B_QM_DIR =", B_QM_DIR)
print("B_PREFIX =", B_PREFIX)
print("B_NPOINTS=", B_NPOINTS)
print("ATOM_C02 =", ATOM_C02, "ATOM_H02 =", ATOM_H02)

In [ ]:
B_INPUT = Path(B_QM_DIR) / "primordia_tim_reaction.input"

if ATOM_C02 is not None and ATOM_H02 is not None:
    text = f"""#RT reaction
#PR gridsize 0 threads 4 band_method BD bandgap 5 eband 1 Rscript
#Reaction dimX {B_NPOINTS} start {B_START}
#Reaction RC1 {ATOM_C02} {ATOM_H02}
#TRJ residues 9 94 164 248
3 {B_PREFIX}{B_QM_EXT} false {B_PREFIX}.pdb mopac 0 0 0 0
"""
    B_INPUT.write_text(text)
    print(text)
    print("Input salvo em:", B_INPUT)
else:
    print("C02/H02 não foram identificados de forma inequívoca. Revise o PDB antes de criar o input.")

### 4B.4 Executar PRIMoRDiA e analisar os outputs

In [ ]:
if B_INPUT.exists():
    first_qm = Path(B_QM_DIR) / f"{B_PREFIX}{B_START}{B_QM_EXT}"
    first_pdb = Path(B_QM_DIR) / f"{B_PREFIX}{B_START}.pdb"
    print("Primeiro QM :", first_qm, first_qm.exists())
    print("Primeiro PDB:", first_pdb, first_pdb.exists())
    if first_qm.exists() and first_pdb.exists():
        cmd = [PRIMORDIA, "-f", B_INPUT.name, "-np", "4", "-verbose"]
        print("$", " ".join(cmd))
        subprocess.run(cmd, cwd=B_QM_DIR, check=True)
    else:
        print("Checkpoint eletrônico/PDB ainda não disponível.")

In [ ]:
if B_INPUT.exists():
    for f in sorted(Path(B_QM_DIR).glob("*.R")):
        print(f.name)
    log = Path(B_QM_DIR) / "primordia.log"
    if log.exists():
        print("\n--- final do log ---")
        print("\n".join(log.read_text(errors="ignore").splitlines()[-40:]))

**Perguntas para discussão da TIM**

1. Como a energia muda ao longo da distância C02–H02?
2. Quais descritores mudam antes, na vizinhança e depois da região de maior energia?
3. Os resíduos acompanhados respondem eletronicamente de forma coordenada?
4. O que o perfil eletrônico acrescenta ao gráfico de energia e à coordenada geométrica?


## 4C. Atividade C — HIV protease: descritores, PCA e KRLS

O repositório contém 12 complexos, 12 ligantes, a proteína de referência e os valores experimentais de energia livre. Os inputs EW e BD já estão preparados.

A análise estatística é **exploratória**: o ajuste KRLS nesta pequena família serve para visualizar tendências no conjunto e não deve ser apresentado como validação de uma função de score.


### 4C.1 Conferir os arquivos

In [ ]:
C_DIR = COURSE / "Atividade_C"
print("complexos AUX.GZ:", len(list(C_DIR.glob("complex_*.aux.gz"))))
print("ligantes AUX.GZ :", len(list(C_DIR.glob("ligand_*.aux.gz"))))
print("protein.aux.gz  :", (C_DIR/"protein.aux.gz").exists())
print("experimental_dG :", (C_DIR/"experimental_dG.txt").exists())
print("\nPrimeiras linhas de experimental_dG.txt:\n")
print("\n".join((C_DIR/"experimental_dG.txt").read_text().splitlines()[:16]))

### 4C.2 Criar diretórios de trabalho EW e BD sem duplicar os arquivos grandes

In [ ]:
import os, shutil

C_WORK = Path("/content/work_C")

def prepare_c_work(name, input_name):
    w = C_WORK / name
    if w.exists():
        shutil.rmtree(w)
    w.mkdir(parents=True)
    # Symlinks para dados pesados; input é copiado para poder ser editado sem alterar o clone.
    for f in C_DIR.iterdir():
        if f.is_file() and f.name != input_name:
            os.symlink(f, w / f.name)
    shutil.copy2(C_DIR / input_name, w / input_name)
    return w

C_EW = prepare_c_work("EW", "primordia_EW.input")
C_BD = prepare_c_work("BD", "primordia_BD.input")
print("EW:", C_EW)
print("BD:", C_BD)

### 4C.3 Executar EW e BD

In [ ]:
RUN_C_EW = True
RUN_C_BD = True

if RUN_C_EW:
    subprocess.run([PRIMORDIA, "-f", "primordia_EW.input", "-verbose"], cwd=C_EW, check=True)
if RUN_C_BD:
    subprocess.run([PRIMORDIA, "-f", "primordia_BD.input", "-verbose"], cwd=C_BD, check=True)

for label, w in [("EW",C_EW),("BD",C_BD)]:
    print("\n", label)
    for name in ["complex_protein_diff.txt", "primordia.log"]:
        print(name, (w/name).exists())

### 4C.4 Inspecionar a matriz e associar com ΔG experimental

In [ ]:
import pandas as pd

def read_dg(path):
    rows = []
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        p = line.split()
        rows.append((p[0], float(p[1])))
    return pd.DataFrame(rows, columns=["id","dG"])

dg = read_dg(C_DIR / "experimental_dG.txt")
display(dg)

for label, w in [("EW",C_EW),("BD",C_BD)]:
    matrix_file = w / "complex_protein_diff.txt"
    if matrix_file.exists():
        m = pd.read_csv(matrix_file, sep=r"\s+")
        display(m.head())
        print(label, "shape=", m.shape)

### 4C.5 PCA + KRLS em R

O script abaixo faz uma versão compacta do fluxo didático:

- associa `complex_protein_diff.txt` ao `experimental_dG.txt`;
- remove variáveis sem variância;
- padroniza;
- executa PCA;
- usa até 5 PCs como entrada do KRLS (`lambda = 0.1`);
- salva gráfico de variância do PCA, PC1×PC2 e observado×ajustado.

**Atenção:** os valores do KRLS são ajustados no mesmo conjunto usado para treinar o modelo. A finalidade aqui é interpretação de tendência, não avaliação preditiva.


In [ ]:
C_R_SCRIPT = Path("/content/analysis_pca_krls_minicurso.R")
C_R_SCRIPT.write_text(r'''suppressPackageStartupMessages({
  library(ggplot2)
  library(KRLS)
})

args <- commandArgs(trailingOnly=TRUE)
if(length(args) < 3) stop("Uso: Rscript script.R matrix.txt experimental_dG.txt prefix")
matrix_file <- args[1]
dg_file <- args[2]
prefix <- args[3]

x <- read.table(matrix_file, header=TRUE, check.names=FALSE)
dg <- read.table(dg_file, comment.char="#", col.names=c("id","dG"), stringsAsFactors=FALSE)

x$id <- sub("^complex_", "", x$frame)
x$id <- sub("\\..*$", "", x$id)
d <- merge(dg, x, by="id")
if(nrow(d) < 5) stop("Poucos complexos associados entre matriz e experimental_dG")

pred_names <- setdiff(names(d), c("id","dG","frame"))
Xdf <- d[, pred_names, drop=FALSE]
num <- vapply(Xdf, is.numeric, logical(1))
Xdf <- Xdf[, num, drop=FALSE]
keep <- vapply(Xdf, function(z) is.finite(sd(z, na.rm=TRUE)) && sd(z, na.rm=TRUE) > 1e-12, logical(1))
Xdf <- Xdf[, keep, drop=FALSE]
X <- scale(Xdf)

pca <- prcomp(X, center=FALSE, scale.=FALSE)
var <- pca$sdev^2 / sum(pca$sdev^2)
var_df <- data.frame(PC=seq_along(var), variance=var)

g1 <- ggplot(var_df, aes(PC, variance)) + geom_col() + geom_line() + geom_point() +
      theme_minimal(base_size=12) + labs(y="Explained variance", title=paste(prefix,"PCA variance"))
ggsave(paste0(prefix,"_pca_variance.png"), g1, width=6, height=4, dpi=300)

scores <- as.data.frame(pca$x)
scores$id <- d$id
scores$dG <- d$dG
if(ncol(pca$x) >= 2) {
  g2 <- ggplot(scores, aes(PC1, PC2, label=id)) + geom_point(size=3) + geom_text(vjust=-0.6) +
        theme_minimal(base_size=12) + labs(title=paste(prefix,"PCA scores"))
  ggsave(paste0(prefix,"_pca_scores.png"), g2, width=6, height=5, dpi=300)
}

npc <- min(5, ncol(pca$x), nrow(d)-1)
Xk <- as.matrix(pca$x[, seq_len(npc), drop=FALSE])
fit <- KRLS::krls(X=Xk, y=d$dG, lambda=0.1, derivative=FALSE, vcov=FALSE)
pred <- as.numeric(fit$fitted)
res <- data.frame(id=d$id, experimental=d$dG, fitted=pred)
write.table(res, paste0(prefix,"_krls_fitted.tsv"), sep="\\t", quote=FALSE, row.names=FALSE)

g3 <- ggplot(res, aes(experimental, fitted, label=id)) + geom_point(size=3) + geom_text(vjust=-0.6) +
      geom_abline(intercept=0, slope=1, linetype=2) + theme_minimal(base_size=12) +
      labs(title=paste(prefix,"KRLS — exploratory in-sample fit"), x="Experimental ΔG", y="KRLS fitted")
ggsave(paste0(prefix,"_krls_fit.png"), g3, width=6, height=5, dpi=300)

cat("N =", nrow(d), "\n")
cat("Predictors after filtering =", ncol(Xdf), "\n")
cat("PCs used in KRLS =", npc, "\n")
cat("KRLS R2 (in-sample) =", fit$R2, "\n")
cat("PCA variance PC1-PC3 =", paste(round(var[seq_len(min(3,length(var)))],4), collapse=", "), "\n")
''')
print(C_R_SCRIPT)

In [ ]:
for label, w in [("EW",C_EW),("BD",C_BD)]:
    matrix_file = w / "complex_protein_diff.txt"
    if matrix_file.exists():
        subprocess.run([
            "Rscript", str(C_R_SCRIPT),
            str(matrix_file), str(C_DIR/"experimental_dG.txt"), label
        ], cwd=w, check=True)
        print("\nOutputs", label)
        for f in sorted(w.glob(f"{label}_*")):
            print(" ", f.name)

### 4C.6 Escolher dois complexos para interpretação local

Uma seleção simples para aula é usar os extremos de ΔG do subconjunto apenas para garantir contraste. A escolha final pode ser substituída por dois complexos mais interessantes quimicamente após examinar PCA/loadings.


In [ ]:
dg_sorted = dg.sort_values("dG")
selected_ids = [dg_sorted.iloc[0]["id"], dg_sorted.iloc[-1]["id"]]
print("Complexos sugeridos pelos extremos de ΔG:", selected_ids)
for pid in selected_ids:
    print(pid, "->", C_DIR / f"complex_{pid}.pdb")

# 5. Visualização 3D no Colab e coloração pelo B-factor

**Sim, é possível fazer isso diretamente no Colab.** O `py3Dmol` usa 3Dmol.js e permite colorir representações de um PDB usando propriedades atômicas. Para PDBs nos quais o PRIMoRDiA/PyMOL armazena um descritor no campo **B-factor**, podemos usar `prop: 'b'` e um gradiente.

Isto é muito útil para **inspeção interativa durante a aula**: girar a estrutura, aproximar o sítio e comparar mapas sem sair do notebook.

Para imagens finais de publicação, o PyMOL continua mais adequado por oferecer controle fino de câmera, ray tracing, transparência, labels e resolução.


### 5.1 Função de visualização colorida pelo B-factor

In [ ]:
import py3Dmol
from pathlib import Path

def pdb_bfactor_range(pdb_path, ignore_zero=False):
    vals = []
    with open(pdb_path, errors="ignore") as fh:
        for line in fh:
            if line.startswith(("ATOM  ", "HETATM")) and len(line) >= 66:
                try:
                    b = float(line[60:66])
                    if ignore_zero and abs(b) < 1e-15:
                        continue
                    vals.append(b)
                except ValueError:
                    pass
    if not vals:
        return 0.0, 1.0
    return min(vals), max(vals)


def view_bfactor_pdb(pdb_path, vmin=None, vmax=None, active_residues=None,
                     gradient="rwb", width=900, height=600,
                     background="white"):
    """Visualiza PDB no Colab usando o B-factor como propriedade de cor.

    active_residues: lista de números de resíduo para destacar em sticks.
    gradient: gradiente aceito pelo 3Dmol.js, por exemplo 'rwb' ou 'roygb'.
    """
    pdb_path = Path(pdb_path)
    if not pdb_path.exists():
        raise FileNotFoundError(pdb_path)

    if vmin is None or vmax is None:
        auto_min, auto_max = pdb_bfactor_range(pdb_path)
        vmin = auto_min if vmin is None else vmin
        vmax = auto_max if vmax is None else vmax
    if vmin == vmax:
        vmax = vmin + 1e-12

    scheme = {"prop": "b", "gradient": gradient, "min": float(vmin), "max": float(vmax)}
    pdb_text = pdb_path.read_text(errors="ignore")

    view = py3Dmol.view(width=width, height=height)
    view.addModel(pdb_text, "pdb")
    # Proteína/macromolécula como cartoon colorido pelo B-factor.
    view.setStyle({}, {"cartoon": {"colorscheme": scheme}})
    # Heteroátomos em sticks; a mesma escala é mantida.
    view.addStyle({"hetflag": True}, {"stick": {"radius": 0.20, "colorscheme": scheme}})

    if active_residues:
        view.addStyle({"resi": [int(x) for x in active_residues]},
                      {"stick": {"radius": 0.18, "colorscheme": scheme}})

    view.setBackgroundColor(background)
    view.zoomTo()
    print(f"B-factor range usado: {vmin:.6g} .. {vmax:.6g}")
    return view.show()


### 5.2 Teste rápido com um PDB já presente no repositório

In [ ]:
test_pdb = C_DIR / "complex_1HSG.pdb"
view_bfactor_pdb(test_pdb, width=850, height=500)

### 5.3 Usar a mesma escala para comparar dois mapas de descritor

Quando dois PDBs contêm valores de descritor no B-factor, **não deixe cada estrutura escolher sua própria escala**. Calcule os limites conjuntos e use os mesmos `vmin`/`vmax`.


In [ ]:
def shared_bfactor_range(*pdb_paths):
    ranges = [pdb_bfactor_range(p) for p in pdb_paths]
    return min(x[0] for x in ranges), max(x[1] for x in ranges)

# Exemplo de uso quando os dois PDBs de descritor estiverem disponíveis:
# pdb1 = Path("frame_inicial_descritor.pdb")
# pdb2 = Path("frame_selecionado_descritor.pdb")
# vmin, vmax = shared_bfactor_range(pdb1, pdb2)
# view_bfactor_pdb(pdb1, vmin=vmin, vmax=vmax, active_residues=[173,176,260,327])
# view_bfactor_pdb(pdb2, vmin=vmin, vmax=vmax, active_residues=[173,176,260,327])


# 6. Checklist final da aula

Ao final do minicurso, verifique se você consegue explicar e reproduzir cada ligação do workflow:

- [ ] estrutura/topologia → preparação com OOCCuPY/pDynamo3;
- [ ] QM/MM ou scan → conjunto de geometrias;
- [ ] refinamento MOPAC → estrutura eletrônica resolvida;
- [ ] estrutura eletrônica + PDB → descritores do PRIMoRDiA;
- [ ] `trajectory` → variação temporal e seleção de frames;
- [ ] `reaction` → descritores ao longo de uma coordenada de reação;
- [ ] proteína–ligante → matriz de mudanças eletrônicas + análise exploratória;
- [ ] B-factor/PyMOL ou `py3Dmol` → representação espacial dos descritores;
- [ ] mesma escala de cor → comparação visual fisicamente consistente.

## Repositórios usados

- Minicurso: `https://github.com/bardenChem/minicurso_xii_emmsb_primordia`
- PRIMoRDiA 1.50: `https://github.com/bardenChem/PRIMoRDiA_1.50v`
- OOCCuPY: `https://github.com/bardenChem/OOCCuPY`
- pDynamo3: `https://github.com/pdynamo/pDynamo3`
- MOPAC: `https://github.com/openmopac/MOPAC`
